In [12]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_columns', None)

In [13]:
# Load datasets
df = pd.read_csv('../../datasets/steam_games_requirements.csv')
cpu_bench = pd.read_csv('../../datasets/raw/CPU_UserBenchmarks.csv')

# Create CPU dictionary
cpu_bench['full_name'] = cpu_bench['Brand'] + ' ' + cpu_bench['Model']
cpu_dict = dict(zip(cpu_bench['full_name'], cpu_bench['Benchmark']))
min_score = cpu_bench['Benchmark'].min()

print(f'Games: {len(df)}')
print(f'CPU benchmarks: {len(cpu_dict)}')
print(f'Lowest benchmark score: {min_score}')

Games: 19875
CPU benchmarks: 1423
Lowest benchmark score: 14.7


In [16]:
def normalize_cpu_name(cpu_str):
    # Normalize: i3 3130 -> i3-3130, Core i3 3130 -> Core i3-3130
    # Match patterns like "i3 3130" or "Core i5 6600K" and add dash
    cpu_str = re.sub(r'([iIcCoRe]+\s*\d)\s+(\d{4})', r'\1-\2', cpu_str)
    # Also handle AMD: Ryzen 5 1600 -> Ryzen 5-1600
    cpu_str = re.sub(r'(Ryzen\s+\d)\s+(\d{4})', r'\1-\2', cpu_str)
    # Handle FX: FX 8320 -> FX-8320
    cpu_str = re.sub(r'(FX)\s+(\d{4})', r'\1-\2', cpu_str)
    # Handle Phenom: Phenom II X3 720 -> Phenom-II-X3-720
    cpu_str = re.sub(r'(Phenom\s*II?)\s+(X\d)\s+(\d{3})', r'\1-\2-\3', cpu_str)
    return cpu_str

def clean_cpu_string(cpu_str):
    if pd.isna(cpu_str):
        return []
        
    cpu_str = str(cpu_str)
    
    # Remove special chars
    cpu_str = cpu_str.replace('®', '').replace('™', '').replace('�', '')
    cpu_str = cpu_str.replace('(r)', '').replace('(tm)', '')
    
    # Normalize i3 3130 -> i3-3130 before splitting
    cpu_str = normalize_cpu_name(cpu_str)
    
    # Split by alternatives
    parts = re.split(r'[/|]', cpu_str)
    
    cleaned = []
    for part in parts:
        part = part.strip()
        
        # Remove common patterns
        removals = [' or better', 'or better', ' or equivalent', 'or equivalent',
                   ' or higher', 'or greater', '64-bit', '(64-bit)',
                   '@ 2.0 GHz', '@ 3.6 GHz', '@ ', 'GHz)', 'GHz ']
        for r in removals:
            part = part.replace(r, '')
        
        # Remove patterns like (2 * 2660)
        part = re.sub(r'\(\d+\s*[*×]\s*\d+\)', '', part)
        part = re.sub(r'\(\d+\.\d+\s*GHz\)', '', part)
        
        part = part.strip()
        
        # Exclude very generic strings
        if part and not any(w in part.lower() for w in ['equivalent', 'anything', 'minimum', 'requires', 'directx']):
            cleaned.append(part)
    
    return cleaned

In [17]:
def normalize_cpu_name(cpu_str):
    # Normalize: i3 3130 -> i3-3130, Core i3 3130 -> Core i3-3130
    cpu_str = re.sub(r'([iIcCoRe]+\s*\d)\s+(\d{4})', r'\1-\2', cpu_str)
    cpu_str = re.sub(r'(Ryzen\s+\d)\s+(\d{4})', r'\1-\2', cpu_str)
    cpu_str = re.sub(r'(FX)\s+(\d{4})', r'\1-\2', cpu_str)
    cpu_str = re.sub(r'(Phenom\s*II?)\s+(X\d)\s+(\d{3})', r'\1-\2-\3', cpu_str)
    return cpu_str

def find_cpu_score_v2(cpu_str, cpu_dict, min_score):
    if pd.isna(cpu_str) or cpu_str == '':
        return None
    
    # Normalize input string
    cpu_str_norm = normalize_cpu_name(cpu_str)
    
    cpus = clean_cpu_string(cpu_str_norm)
    scores = []
    matched_any = False
    
    for cpu in cpus:
        cpu = cpu.strip()
        if not cpu:
            continue
        
        # Exact match
        if cpu in cpu_dict:
            scores.append(cpu_dict[cpu])
            matched_any = True
        else:
            # Try normalized benchmark names too
            for bench_cpu, score in cpu_dict.items():
                bench_norm = normalize_cpu_name(bench_cpu)
                if cpu.lower() in bench_norm.lower() or bench_norm.lower() in cpu.lower():
                    scores.append(score)
                    matched_any = True
                    break
    
    if scores:
        return min(scores)
    elif matched_any == False and cpu_str:
        return min_score
    return None

In [18]:
# Apply scoring
df['min_cpu_score'] = df['min_cpu'].apply(lambda x: find_cpu_score_v2(x, cpu_dict, min_score))
df['rec_cpu_score'] = df['rec_cpu'].apply(lambda x: find_cpu_score_v2(x, cpu_dict, min_score))

# Results
total = len(df)
matched_min = df['min_cpu_score'].notna().sum()
matched_rec = df['rec_cpu_score'].notna().sum()

print(f'Total games: {total}')
print(f'Matched min_cpu: {matched_min} ({matched_min/total*100:.1f}%)')
print(f'Matched rec_cpu: {matched_rec} ({matched_rec/total*100:.1f}%)')

Total games: 19875
Matched min_cpu: 19875 (100.0%)
Matched rec_cpu: 18053 (90.8%)


In [20]:
# Show sample
df.head(5)

,app_id,name,genre,min_os,min_cpu,min_ram,min_gpu,rec_os,rec_cpu,rec_ram,rec_gpu,min_cpu_score,rec_cpu_score
0,379720,DOOM,Action,Windows 7/8.1/10 (64-bit versions),Intel Core i5-2400/AMD FX-8320 or better,8 GB RAM,NVIDIA GTX 670 2GB/AMD Radeon HD 7870 2GB or b...,Windows 7/8.1/10 (64-bit versions),Intel Core i7-3770/AMD FX-8350 or better,8 GB RAM,NVIDIA GTX 970 4GB/AMD Radeon R9 290 4GB or be...,58.5,60.8
1,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,"Action,Adventure,Massively Multiplayer",64-bit Windows 7,Intel Core i5-4430 / AMD FX-6300,8 GB RAM,NVIDIA GeForce GTX 960 2GB / AMD Radeon R7 370...,64-bit Windows 7,Intel Core i5-6600K / AMD Ryzen 5 1600,16 GB RAM,NVIDIA GeForce GTX 1060 3GB / AMD Radeon RX 58...,55.2,70.2
2,637090,BATTLETECH,"Action,Adventure,Strategy",64-bit Windows 7 or Higher,Intel® Core™ i3-2105 or AMD® Phenom™ II X3 720,8 GB RAM,Nvidia® GeForce™ GTX 560 Ti or AMD® ATI Radeon...,64-bit Windows 7 or Higher,Intel® Core™ i5-4460 or AMD® FX-4300,16 GB RAM,Nvidia® GeForce™ GTX 670 or AMD® Radeon™ R9 28...,58.5,62.1
3,221100,DayZ,"Action,Adventure,Massively Multiplayer",Windows 7/8.1 64-bit,Intel Core i5-4430,8 GB RAM,NVIDIA GeForce GTX 760 or AMD R9 270X,Windows 10 64-bit,Intel Core i5-6600K or AMD R5 1600X,12 GB RAM,NVIDIA GeForce GTX 1060 or AMD RX 580,65.2,72.4
4,8500,EVE Online,"Action,Free to Play,Massively Multiplayer,RPG,...",Windows 7,Intel Dual Core @ 2.0 GHz,2 GB,NaN,Windows 10,Intel i7-7700 or AMD Ryzen 7 1700 @ 3.6 GHz or...,16 GB or greater,NaN,14.7,69.0
